In [1]:
# Importing necessary libraries
import pandas as pd
import os
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from datasets import Dataset
from transformers import pipeline
import joblib

# Set the directory path where your CSV files are located
directory = r"C:\Users\muham\OneDrive\Documents\Semester7\FYP\intentsnewdataset\dataset"


c:\Users\muham\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# List of CSV files
csv_files = [
    "interviewer_bot_affirm.csv",
    "interviewer_bot_deny.csv",
    "interviewer_bot_greetings.csv",
    "interviewer_bot_inform_name.csv",
    "interviewer_bot_organization_inquiries.csv",
    "interviewer_bot_salary.csv",
    "interviewer_bot_uncertain.csv",
    "new_education_samples.csv",
    "new_experience_samples.csv",
    "new_inform_address_samples.csv",
    "new_skills_samples.csv"
]

# Initialize an empty DataFrame to hold all the data
all_data = pd.DataFrame()

# Load and combine all CSV files into one DataFrame
for file in csv_files:
    file_path = os.path.join(directory, file)
    data = pd.read_csv(file_path)
    all_data = pd.concat([all_data, data], ignore_index=True)

# Preprocess the data
X = all_data['Sample'].tolist()  # List of text samples
y = all_data['Intent'].astype('category').cat.codes.tolist()  # Convert Intent to category codes

# Split the dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Load the DistilBERT tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Tokenize the dataset
def tokenize_function(texts):
    return tokenizer(texts, padding="max_length", truncation=True)

# Convert to Hugging Face Dataset object
train_data = Dataset.from_dict({'text': X_train, 'labels': y_train}).map(lambda e: tokenize_function(e['text']), batched=True)
test_data = Dataset.from_dict({'text': X_test, 'labels': y_test}).map(lambda e: tokenize_function(e['text']), batched=True)

# Load the DistilBERT model
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=len(set(y)))

# Training arguments
training_args = TrainingArguments(
    output_dir='./results',              # Output directory
    evaluation_strategy="epoch",         # Evaluate every epoch
    learning_rate=2e-5,                  # Learning rate
    per_device_train_batch_size=16,      # Batch size for training
    per_device_eval_batch_size=16,       # Batch size for evaluation
    num_train_epochs=3,                  # Number of epochs
    weight_decay=0.01,                   # Weight decay
)

# Define a compute_metrics function to calculate accuracy
def compute_metrics(pred):
    predictions, labels = pred
    preds = predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

# Save the model and tokenizer
model.save_pretrained("./distilbert-intent-classification")
tokenizer.save_pretrained("./distilbert-intent-classification")
# Evaluate the model on the test set
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

# Save the trained model using joblib
joblib.dump(trainer, 'distilbert_trainer.pkl')



Map: 100%|██████████| 994/994 [00:00<00:00, 1397.04 examples/s]
c:\Users\muham\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\muham\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of DistilB

{'eval_loss': 0.04619395732879639, 'eval_accuracy': 0.9949698189134809, 'eval_runtime': 330.776, 'eval_samples_per_second': 3.005, 'eval_steps_per_second': 0.19, 'epoch': 1.0}


                                                          
 67%|██████▋   | 498/747 [12:59:56<1:29:07, 21.48s/it]

{'eval_loss': 0.01919395849108696, 'eval_accuracy': 0.9969818913480886, 'eval_runtime': 428.1745, 'eval_samples_per_second': 2.321, 'eval_steps_per_second': 0.147, 'epoch': 2.0}


 67%|██████▋   | 500/747 [13:00:40<7:39:10, 111.54s/it] 

...

                                                       
100%|██████████| 747/747 [14:36:38<00:00, 70.41s/it]


{'eval_loss': 0.012417097575962543, 'eval_accuracy': 0.9969818913480886, 'eval_runtime': 346.4245, 'eval_samples_per_second': 2.869, 'eval_steps_per_second': 0.182, 'epoch': 3.0}
{'train_runtime': 52598.6114, 'train_samples_per_second': 0.227, 'train_steps_per_second': 0.014, 'train_loss': 0.2304240230575623, 'epoch': 3.0}


100%|██████████| 63/63 [05:35<00:00,  5.32s/it]


Evaluation results: {'eval_loss': 0.012417097575962543, 'eval_accuracy': 0.9969818913480886, 'eval_runtime': 340.491, 'eval_samples_per_second': 2.919, 'eval_steps_per_second': 0.185, 'epoch': 3.0}


TypeError: cannot pickle 'tensorflow.python.lib.io._pywrap_file_io.WritableFile' object

In [12]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

# Load the saved model and tokenizer
model = DistilBertForSequenceClassification.from_pretrained("./distilbert-intent-classification")
tokenizer = DistilBertTokenizer.from_pretrained("./distilbert-intent-classification")

# Perform inference or further tasks


In [12]:
# Importing necessary libraries
import pandas as pd
import os
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from datasets import Dataset
from transformers import pipeline
import joblib

# Set the directory path where your CSV files are located
directory = r"C:\Users\muham\OneDrive\Documents\Semester7\FYP\intentsnewdataset\dataset"

# Define a function to load data and return intent labels
def load_data_and_intent_labels(directory):
    # List of CSV files
    csv_files = [
        "interviewer_bot_affirm.csv",
        "interviewer_bot_deny.csv",
        "interviewer_bot_greetings.csv",
        "interviewer_bot_inform_name.csv",
        "interviewer_bot_organization_inquiries.csv",
        "interviewer_bot_salary.csv",
        "interviewer_bot_uncertain.csv",
        "new_education_samples.csv",
        "new_experience_samples.csv",
        "new_inform_address_samples.csv",
        "new_skills_samples.csv"
    ]
    
    # Initialize an empty DataFrame to hold all the data
    all_data = pd.DataFrame()

    # Load and combine all CSV files into one DataFrame
    for file in csv_files:
        file_path = os.path.join(directory, file)
        data = pd.read_csv(file_path)
        all_data = pd.concat([all_data, data], ignore_index=True)

    # Extract intent labels
    intent_labels = all_data['Intent'].astype('category').cat.categories.tolist()
    return intent_labels


In [13]:
print(load_data_and_intent_labels(directory))

['Affirm', 'Deny', 'Greeting', 'Inform Salary', 'Inform_Name', 'Organization_inquiry', 'Uncertain', 'education', 'experience', 'inform_address', 'skills']


In [1]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch
# Load the saved model and tokenizer
model = DistilBertForSequenceClassification.from_pretrained("./distilbert-intent-classification")
tokenizer = DistilBertTokenizer.from_pretrained("./distilbert-intent-classification")

# Load intent labels using the function
# intent_labels = load_data_and_intent_labels(directory)
intent_labels=['Affirm', 'Deny', 'Greeting', 'Inform Salary', 'Inform_Name', 'Organization_inquiry', 'Uncertain', 'education', 'experience', 'inform_address', 'skills']
# Example text input for inference
text_samples = ["ok"]

# Tokenize the input texts
inputs = tokenizer(text_samples, padding=True, truncation=True, return_tensors="pt")

# Perform inference (get model predictions)
with torch.no_grad():
    outputs = model(**inputs)

# Get predicted labels (the label with the highest score for each input)
predictions = torch.argmax(outputs.logits, dim=-1)

# Print predictions
for i, prediction in enumerate(predictions):
    # Convert tensor to integer index
    pred_index = prediction.item()  # Get the integer value from the tensor
    print(f"Text: {text_samples[i]}")
    print(f"Predicted Intent: {intent_labels[pred_index]}\n")


c:\Users\muham\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 

In [24]:
# Generate a classification report
report = classification_report(y_test, y_pred, target_names=intent_labels)
print(report)

                      precision    recall  f1-score   support

              Affirm       0.97      1.00      0.98        86
                Deny       1.00      1.00      1.00        84
            Greeting       1.00      1.00      1.00       104
       Inform Salary       1.00      0.99      0.99        75
         Inform_Name       1.00      1.00      1.00        96
Organization_inquiry       1.00      1.00      1.00        93
           Uncertain       1.00      0.97      0.99        68
           education       1.00      1.00      1.00       100
          experience       1.00      1.00      1.00        99
      inform_address       1.00      1.00      1.00        82
              skills       1.00      1.00      1.00       107

            accuracy                           1.00       994
           macro avg       1.00      1.00      1.00       994
        weighted avg       1.00      1.00      1.00       994

